In [ ]:
# Momentum strategy on S&P 500 — notebook assembly
# Parameters
START_DATE = "2015-01-02"
END_DATE   = "2025-01-01"
UNIVERSE_ASOF = "2015-01-01"  # point-in-time S&P500 membership anchor
TOP_N = 50
BOTTOM_N = 50

import pandas as pd
import numpy as np

from findata.sp500_composition import get_sp500_composition
from findata.equity_prices import get_equity_prices

# Display versions for reproducibility
import sys
print({
    "python": sys.version.split()[0],
})

In [ ]:
# 1) Universe: point-in-time S&P 500 tickers
sp500_tickers = get_sp500_composition(UNIVERSE_ASOF, return_dataframe=False)
print(f"Universe size as of {UNIVERSE_ASOF}: {len(sp500_tickers)} tickers")

# Guard: ensure non-empty
aassert_msg = "No S&P 500 tickers returned; check network/cache for findata.sp500_composition"
assert len(sp500_tickers) > 0, aassert_msg

In [ ]:
# 2) Fetch daily adjusted Close prices for the universe
# findata.get_equity_prices returns MultiIndex columns (field, ticker)
prices_all = get_equity_prices(
    tickers=sp500_tickers,
    start_date=START_DATE,
    end_date=END_DATE,
    fields=["Close"],
    frequency="1d",
    auto_adjust=True,
)
close = prices_all["Close"].copy()  # wide [dates x tickers]

# Basic cleaning: drop columns entirely NaN and rows with all-NaN
close = close.dropna(axis=1, how="all").dropna(axis=0, how="all")
print(f"Close shape: {close.shape}")

In [ ]:
# 3) Compute simple daily asset returns
asset_returns = close.pct_change().replace([np.inf, -np.inf], np.nan)

# Optional: keep dates where at least some returns are available
asset_returns = asset_returns.dropna(how="all")

# Sanity check
asset_returns.info(verbose=False)
asset_returns.tail(3)

In [ ]:
# 4) Momentum signal: z-score of 30-day average returns (cross-sectional per date)
# Compute 30D average of daily returns
avg30 = asset_returns.rolling(window=30, min_periods=20).mean()

# Cross-sectional z-score each date: (x - mean)/std across tickers
cs_mean = avg30.mean(axis=1)
cs_std = avg30.std(axis=1).replace(0.0, np.nan)
signal_z = (avg30.sub(cs_mean, axis=0)).div(cs_std, axis=0)

# Clip extreme z-scores for stability (optional)
signal_z = signal_z.clip(lower=-5, upper=5)

signal_z.tail(3)

In [ ]:
# 5) Construct daily long-short portfolio: top 50 long, bottom 50 short (equal-weight), dollar neutral

def build_long_short_weights(scores: pd.DataFrame, top_n: int, bottom_n: int) -> pd.DataFrame:
    # For each date, pick top and bottom names by score; ignore NaNs.
    def one_day_weights(row: pd.Series) -> pd.Series:
        s = row.dropna()
        if s.empty:
            return pd.Series(index=row.index, dtype=float)
        # Ranks: highest = long
        longs = s.nlargest(min(top_n, len(s))).index
        shorts = s.nsmallest(min(bottom_n, len(s))).index
        w = pd.Series(0.0, index=row.index)
        if len(longs) > 0:
            w.loc[longs] = +0.5 / len(longs)
        if len(shorts) > 0:
            w.loc[shorts] = -0.5 / len(shorts)
        return w

    weights = scores.apply(one_day_weights, axis=1)
    # Ensure NaNs -> 0
    weights = weights.fillna(0.0)
    # Remove days with no positions
    weights = weights.loc[(weights.abs().sum(axis=1) > 0)]
    return weights

asset_weights_raw = build_long_short_weights(signal_z, TOP_N, BOTTOM_N)
asset_weights_raw.tail(3)

In [ ]:
# 6) Apply one-day lag to weights to avoid lookahead bias and align with returns
asset_weights = asset_weights_raw.reindex(asset_returns.index).reindex(columns=asset_returns.columns)
asset_weights = asset_weights.shift(1).fillna(0.0)

# Optional: ensure gross exposure = 1.0 where non-zero
gross = asset_weights.abs().sum(axis=1)
scaler = pd.Series(1.0, index=asset_weights.index)
nonzero = gross > 0
scaler.loc[nonzero] = 1.0 / gross.loc[nonzero]
asset_weights = asset_weights.mul(scaler, axis=0)

# Final shapes
print("asset_returns:", asset_returns.shape)
print("asset_weights:", asset_weights.shape)
asset_weights.tail(3)

In [ ]:
# 7) Compute daily strategy returns from weights and asset returns (for diagnostics)
strategy_returns = (asset_weights * asset_returns).sum(axis=1)

print("Preview strategy returns:")
strategy_returns.tail(5)

In [ ]:
(strategy_returns+1).cumprod().plot()

In [ ]:
# 8) Final: call the user-defined backtest function exactly once
# Expecting a function `backtest(asset_returns: pd.DataFrame, asset_weights: pd.DataFrame)` to be defined by the user.
def backtest(asset_returns, asset_weights):
    raise Exception(NotImplementedError)
    
backtest_results = backtest(asset_returns=asset_returns, asset_weights=asset_weights)
backtest_results